In [ ]:
from google.colab import drive
import os

print("Mounting Google Drive...")
drive.mount('/content/drive')

project_path = "/content/drive/My Drive/meme-analysis-ds"

print(f"Changing directory to: {project_path}")
os.chdir(project_path)

print("\nFiles in project folder:")
!ls

Mounting Google Drive...
Mounted at /content/drive
Changing directory to: /content/drive/My Drive/meme-analysis-ds

Files in project folder:
images	    labels.gsheet		train.ipynb
labels.csv  my-awesome-meme-classifier	wandb


In [ ]:
import sys
!{sys.executable} -m pip install --upgrade transformers
!{sys.executable} -m pip install datasets accelerate torch pillow -q
!{sys.executable} -m pip install pandas -q
!{sys.executable} -m pip install evaluate -q

import os
import zipfile
import pandas as pd
import torch
import numpy as np
import evaluate
from datasets import Dataset, Image, DatasetDict, Features, ClassLabel
from transformers import ViTImageProcessor, ViTForImageClassification
from transformers import TrainingArguments, Trainer

from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

print("All libraries installed, upgraded, and imported.")

zip_file_name = "archive.zip"
expected_image_folder = "images/images"
expected_csv_file = "labels.csv"

if os.path.exists(expected_image_folder) and os.path.exists(expected_csv_file):
    print("Dataset already unzipped. Skipping extraction.")
else:

    if not os.path.exists(zip_file_name):
        print(f"ERROR: Zip file '{zip_file_name}' not found.")
        print("Please download the '6992 Labeled Meme Images Dataset' zip file,")
        print("move it to this folder, and make sure it is named 'archive.zip'.")
        raise SystemExit("Zip file not found.")

    print(f"Extracting '{zip_file_name}'... This may take a moment.")
    with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
        zip_ref.extractall(".")

    print("Extraction complete. The 'images' folder and 'labels.csv' are ready.")

csv_file_path = "labels.csv"
image_dir_path = "images/images" # <-- This was your fix
print(f"Successfully found dataset at: {image_dir_path} and {csv_file_path}")

print("Loading CSV file...")
df = pd.read_csv(csv_file_path)

df = df.rename(columns={"image_name": "File_Name", "overall_sentiment": "Label"})

print(f"Original 'File_Name' column type: {df['File_Name'].dtype}")
df["File_Name"] = df["File_Name"].astype(str)
print(f"New 'File_Name' column type: {df['File_Name'].dtype}")

df = df.dropna(subset=["Label", "File_Name"])
print(f"Original CSV rows: {len(df)}")

df["image_path"] = df["File_Name"].apply(lambda x: os.path.join(image_dir_path, x))

print("Verifying image files...")
df['file_exists'] = df['image_path'].apply(lambda x: os.path.exists(x))
df = df[df['file_exists'] == True]
print(f"Found {len(df)} matching images.")

unique_labels = sorted(df["Label"].unique())
print(f"Found labels: {unique_labels}")

class_label_feature = ClassLabel(names=unique_labels)

# Create the dataset
full_dataset = Dataset.from_pandas(df)

# Create the 'label_id' column
full_dataset = full_dataset.map(lambda examples: {
    "label_id": class_label_feature.str2int(examples["Label"])
})

full_dataset = full_dataset.cast_column("label_id", class_label_feature)

full_dataset = full_dataset.cast_column("image_path", Image(decode=True))
full_dataset = full_dataset.rename_column("image_path", "image")
print("Dataset successfully loaded and processed.")
print(f"\nDataset features: {full_dataset.features}")

dataset_splits = full_dataset.train_test_split(test_size=0.2, stratify_by_column="label_id")
train_dataset = dataset_splits["train"]
test_dataset = dataset_splits["test"]
print(f"Training images: {len(train_dataset)}, Testing images: {len(test_dataset)}")

model_name = "google/vit-base-patch16-224-in21k"
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for i, label in enumerate(unique_labels)}

processor = ViTImageProcessor.from_pretrained(model_name, do_convert_rgb=True)
model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=len(unique_labels),
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True
)

def preprocess_images(examples):
    examples['pixel_values'] = processor(examples['image'], return_tensors="pt").pixel_values

    examples['labels'] = examples['label_id']
    return examples

train_dataset.set_transform(preprocess_images)
test_dataset.set_transform(preprocess_images)

def collate_fn(examples):

    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    label_ids = torch.tensor([example["labels"] for example in examples])
    return {"pixel_values": pixel_values, "labels": label_ids}
print("Model and preprocessor loaded.")

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

model_output_dir = "my-awesome-meme-classifier"

training_args = TrainingArguments(
    output_dir=model_output_dir,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=5,
    learning_rate=2e-5,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=processor,
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
)

print("Starting model training...")
trainer.train()
print("Training finished.")

print("Saving the fine-tuned model...")
trainer.save_model(model_output_dir)
processor.save_pretrained(model_output_dir)

print(f"Model saved! Your trained model is in the folder: '{model_output_dir}'")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00
All libraries installed, upgraded, and imported.
Dataset already unzipped. Skipping extraction.
Successfully found dataset at: images/images and labels.csv
Loading CSV file...
Original 'File_Name' column type: object
New 'File_Name' column type: object
Original CSV rows: 6992
Verifying image files...
Found 6976 matching images.
Found labels: ['negative', 'neutral', 'positive', 'very_negative', 'very_positive']


Map:   0%|          | 0/6976 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/6976 [00:00<?, ? examples/s]

Dataset successfully loaded and processed.

Dataset features: {'Unnamed: 0': Value('int64'), 'File_Name': Value('string'), 'text_ocr': Value('string'), 'text_corrected': Value('string'), 'Label': Value('string'), 'image': Image(mode=None, decode=True), 'file_exists': Value('bool'), '__index_level_0__': Value('int64'), 'label_id': ClassLabel(names=['negative', 'neutral', 'positive', 'very_negative', 'very_positive'])}
Training images: 5580, Testing images: 1396


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: likhiikm0903 (likhiikm0903-rv-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy
1,1.291800,1.276690,0.446991
2,1.278100,1.281471,0.444126
3,1.251500,1.287903,0.446275
4,1.111800,1.297527,0.427650
5,1.160300,1.302374,0.421203


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Training finished.
Saving the fine-tuned model...
Model saved! Your trained model is in the folder: 'my-awesome-meme-classifier'
